# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [1]:
import base64
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _clean_repo_url(repo_url: str) -> str:
    parsed = urlsplit(repo_url)
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, netloc, parsed.path, parsed.query, parsed.fragment))

def _git_clone_env():
    token = os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN') or ''
    env = os.environ.copy()
    if not token:
        return env
    auth_value = base64.b64encode(f'x-access-token:{token}'.encode('utf-8')).decode('ascii')
    config_count = int(str(env.get('GIT_CONFIG_COUNT', '0') or '0'))
    env['GIT_CONFIG_COUNT'] = str(config_count + 1)
    env[f'GIT_CONFIG_KEY_{config_count}'] = 'http.https://github.com/.extraheader'
    env[f'GIT_CONFIG_VALUE_{config_count}'] = f'AUTHORIZATION: basic {auth_value}'
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GCM_INTERACTIVE'] = 'Never'
    return env

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _clean_repo_url(REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)], check=True, env=_git_clone_env())
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())

[BOOTSTRAP] GitHub token found.
[BOOTSTRAP] HuggingFace token found.
[BOOTSTRAP] Repo root: /content/bitirmeprojesi
[BOOTSTRAP] Installing requirements_colab.txt...
[BOOTSTRAP] Notebook 2: Continual Adapter Training bootstrap complete.

BOOTSTRAP STATUS
Status: ok
Repo Root: /content/bitirmeprojesi
In Colab: True
GitHub Token: ✓
HuggingFace Token: ✓

[SETUP] Checking model access for training...
[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.
[KONTROL] Varsayilan backbone modeli: facebook/dinov3-vitl16-pretrain-lvd1689m


In [2]:
# Notebook 2 calisma kimligi
# Final crop/part kimligi parametre hucredeki ADAPTER_KEY ile tekrar senkronize edilir.
CROP_NAME = "strawberry"
PART_NAME = "leaf"
ENABLE_BAYESIAN_OPTIMIZATION = False  # Bayesian optimization devre disi
print(f"[RUN] crop={CROP_NAME} part={PART_NAME} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}")


[RUN] crop=strawberry part=leaf bayes_opt=False


In [3]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())

[BOOTSTRAP] run_id=strawberry_leaf_2026-05-06_06-00-15 crop=strawberry part=leaf


In [4]:
# Tek degistirilecek alan: egitmek istedigin adapter anahtari.
ADAPTER_KEY = "strawberry__leaf"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf

ADAPTER_RECS = {
    "grape__fruit": {
        "crop": "grape", "part": "fruit",
        "ood": "data/prepared_runtime_datasets/grape__fruit/ood",
        "oe": "data/prepared_runtime_datasets/grape__fruit/oe",
        "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False,
        "overrides": {"EPOCHS": 32, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0},
    },
    "grape__leaf": {
        "crop": "grape", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/grape__leaf/ood",
        "oe": "data/prepared_runtime_datasets/grape__leaf/oe",
        "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False,
        "overrides": {"EPOCHS": 28, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.0},
    },
    "strawberry__fruit": {
        "crop": "strawberry", "part": "fruit",
        "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood",
        "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe",
        "oe_enabled": True, "oe_w": 0.10, "allow_under_min": True,
        "overrides": {"EPOCHS": 36, "BATCH_SIZE": 32, "LEARNING_RATE": 8e-5, "LORA_R": 16, "LORA_ALPHA": 16, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 3.0},
    },
    "strawberry__leaf": {
        "crop": "strawberry", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood",
        "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe",
        "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False,
        "overrides": {"EPOCHS": 22, "BATCH_SIZE": 96, "LEARNING_RATE": 1.5e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0},
    },
    "tomato__fruit": {
        "crop": "tomato", "part": "fruit",
        "ood": "data/prepared_runtime_datasets/tomato__fruit/ood",
        "oe": "data/prepared_runtime_datasets/tomato__fruit/oe",
        "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False,
        "overrides": {"EPOCHS": 30, "BATCH_SIZE": 64, "LEARNING_RATE": 8e-5, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0},
    },
    "tomato__leaf": {
        "crop": "tomato", "part": "leaf",
        "ood": "data/prepared_runtime_datasets/tomato__leaf/ood",
        "oe": "data/prepared_runtime_datasets/tomato__leaf/oe",
        "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False,
        "overrides": {"EPOCHS": 20, "BATCH_SIZE": 96, "LEARNING_RATE": 1.2e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0},
    },
}

if ADAPTER_KEY not in ADAPTER_RECS:
    raise ValueError(f"Unsupported ADAPTER_KEY={ADAPTER_KEY!r}. Options: {sorted(ADAPTER_RECS)}")

_adapter_rec = ADAPTER_RECS[ADAPTER_KEY]
CROP_NAME = _adapter_rec["crop"]
PART_NAME = _adapter_rec["part"]
DATASET_NAME = ADAPTER_KEY

# Cell 3, bu parametre hucreden once calistigi icin run kimligini burada dogru adaptere yeniden bagla.
if "ColabLiveTelemetry" in globals() and "TrainingCheckpointManager" in globals():
    RUN_ID = build_notebook_run_id(CROP_NAME, PART_NAME)
    NOTEBOOK_FILENAME = "2_train_continual_sd_lora_adapter.executed.ipynb"
    REPO_RUN_DIR = build_notebook_run_dir(ROOT, CROP_NAME, PART_NAME, RUN_ID)
    REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / "notebooks" / NOTEBOOK_FILENAME
    LOCAL_OUTPUT_DIR = ROOT / "outputs" / "colab_notebook_training"
    REPO_OUTPUT_DIR = REPO_RUN_DIR / "outputs" / "colab_notebook_training"
    REPO_TELEMETRY_DIR = REPO_RUN_DIR / "telemetry"
    REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / "checkpoint_state"
    LOCAL_TELEMETRY_ROOT = ROOT / "outputs" / "colab_notebook_training" / "telemetry_runtime"
    LOCAL_TELEMETRY_SPOOL_ROOT = ROOT / ".runtime_tmp" / "colab_notebook_training" / "telemetry_spool"
    TELEMETRY = ColabLiveTelemetry(
        notebook_name="2_train_continual_sd_lora_adapter.ipynb",
        run_id=RUN_ID,
        drive_root=LOCAL_TELEMETRY_ROOT,
        local_root=LOCAL_TELEMETRY_SPOOL_ROOT,
    )
    CHECKPOINT_ROOT = TELEMETRY.drive_run_dir
    CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    TELEMETRY.configure_repo_output_export(
        notebook_path=REPO_NOTEBOOK_OUTPUT_PATH,
        export_notebook_fn=export_current_colab_notebook,
    )
    TELEMETRY.update_latest({"phase": "adapter_key_selected", "run_id": RUN_ID, "adapter_key": ADAPTER_KEY, "crop": CROP_NAME, "part": PART_NAME})

with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri ---
    # Bu hucreyi duzenleyin, sonra kalan hucreleri sirayla calistirin.
    # Kosu kimligi icin CROP_NAME/PART_NAME degerlerini ustteki hucreden yonetin.

    # RUNTIME_DATASET_ROOT: Notebook 0'un yazdigi <dataset_key>/continual|val|test|ood yapisini tutan repo-ici root.
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"

    # DATASET_NAME: Notebook 0'un urettigi runtime dataset klasor adi. Bos ise notebook kullaniciya sorar.
    DATASET_NAME = ""

    # OOD_ROOT: Gercek OOD klasoru. Bos ise ASK_FOR_OOD_ROOT=True iken notebook yol sorar; Enter varsa runtime ood/ kullanir.
    OOD_ROOT = ""
    ASK_FOR_OOD_ROOT = True

    # OE_ROOT: Outlier Exposure egitim klasoru. OE_ENABLED=True ve bos ise ASK_FOR_OE_ROOT=True iken notebook yol sorar; Enter varsa runtime oe/ kullanir.
    OE_ROOT = ""
    ASK_FOR_OE_ROOT = True
    OE_ENABLED = False
    OE_LOSS_WEIGHT = 0.5

    # CROP_NAME ve PART_NAME, kosu adlandirmasi ve metadata icin kullanilir.
    CROP_NAME = globals().get("CROP_NAME", "tomato")
    PART_NAME = globals().get("PART_NAME", "unspecified")
    ENABLE_BAYESIAN_OPTIMIZATION = bool(globals().get("ENABLE_BAYESIAN_OPTIMIZATION", False))

    # ALLOW_UNDER_MIN_TRAINING: True olursa 100 image/class production guardrail'i research kosulari icin bypass edilir.
    ALLOW_UNDER_MIN_TRAINING = False

    # EPOCHS: train split uzerinden kac tam gecis yapilacagi.
    EPOCHS = 12

    # BATCH_SIZE: optimizer adimi basina ornek sayisi. GPU limitine yakin olacak sekilde artirilabilir.
    BATCH_SIZE = 96

    # LEARNING_RATE: adapter/LoRA parametreleri icin optimizer adim buyuklugu.
    LEARNING_RATE = 2e-4

    # LORA_R: LoRA rank degeri. Buyudukce kapasite ve VRAM/islem maliyeti artar.
    LORA_R = 24

    # LORA_ALPHA: LoRA olcekleme katsayisi. Genelde LORA_R degerinin 2x-4x araliginda kullanilir.
    LORA_ALPHA = 24

    # LORA_DROPOUT: LoRA katmanlarina uygulanan dropout. Buyudukce regularizasyon artar.
    LORA_DROPOUT = 0.1

    # OOD_FACTOR: OOD esik hassasiyetini carpansal olarak ayarlar.
    OOD_FACTOR = 3.0
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: eski/yeni sinif enerji ayrimi icin deneysel egitim regularizeri.
    BER_ENABLED = False

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: eski ve yeni sinif kisimlari icin BER ceza agirliklari.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW weight decay degeri.
    WEIGHT_DECAY = 0.01

    # MIXED_PRECISION: {'off', 'auto', 'fp16', 'bf16'} seceneklerinden biri.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation katsayisi.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping esigi. 0 olursa clipping kapanir.
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing katsayisi.
    LABEL_SMOOTHING = 0.0

    # LOSS_NAME / LOGITNORM_TAU: varsayilan kayip LogitNorm'dur; CE icin loss_name='cross_entropy' secin.
    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    # Scheduler ayarlari.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping ayarlari.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    # Tekrarlanabilirlik ve runtime ayarlari.
    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    # NUM_WORKERS: dataloader worker sayisi. CPU veri yukleme paralelligini belirler.
    NUM_WORKERS = 12

    # PREFETCH: NUM_WORKERS > 0 iken worker basina prefetch katsayisi.
    PREFETCH = 8

    # PIN_MEMORY: host memory sabitleyerek host-to-GPU aktarimini hizlandirir.
    PIN_MEMORY = True

    # USE_CACHE: destekleniyorsa decode edilmis ornekleri RAM'de tutar.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: continual/train split'ini de cache'ler. Yuksek RAM'li Colab icin uygundur.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: her N epoch'ta tam validation calistirir; final epoch her zaman dahildir.
    VALIDATION_EVERY_N_EPOCHS = 1

    # CHECKPOINT_EVERY_N_STEPS / CHECKPOINT_ON_EXCEPTION: notebook checkpoint sikligi ayarlari.
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True

    # STDOUT_BATCH_INTERVAL: canli training ilerleme yazdirma araligi.
    STDOUT_BATCH_INTERVAL = 12

    # RESUME_MODE: "fresh" yeni kosu baslatir, "resume" son checkpointten devam eder.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # AUTO_DISCONNECT_RUNTIME: tum final exportlar basariliysa Colab runtime'i kapatir.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: son durum gorunsun diye disconnect oncesi kisa bekleme suresi.
    AUTO_DISCONNECT_GRACE_SECONDS = 20

    # AUTO_PUSH_TO_GITHUB: final exportlar bitince runs/<crop>/<part>/<RUN_ID>/ klasorunu repoya commit edip pushlar.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: auto-push aciksa kullanilacak git remote adi.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: auto-push icin branch override degeri. None olursa mevcut branch kullanilir.
    AUTO_PUSH_BRANCH = None

    # ADAPTER_KEY secimi bu noktada tum notebook degerlerini kesin olarak ezer.
    rec = ADAPTER_RECS[ADAPTER_KEY]
    CROP_NAME = rec["crop"]
    PART_NAME = rec["part"]
    DATASET_NAME = ADAPTER_KEY
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets"
    OOD_ROOT = rec["ood"]
    ASK_FOR_OOD_ROOT = False
    OE_ROOT = rec["oe"]
    ASK_FOR_OE_ROOT = False
    OE_ENABLED = bool(rec["oe_enabled"])
    OE_LOSS_WEIGHT = float(rec["oe_w"])
    ALLOW_UNDER_MIN_TRAINING = bool(rec["allow_under_min"])
    VALIDATION_EVERY_N_EPOCHS = 1

    MANUAL_PARAM_OVERRIDES = {}
    MANUAL_PARAM_OVERRIDES = {
        **rec["overrides"],
        "WEIGHT_DECAY": 0.01,
        "LOSS_NAME": "logitnorm",
        "LOGITNORM_TAU": 1.0,
        "MIXED_PRECISION": "bf16",
        "GRAD_ACCUM_STEPS": 1,
        "VALIDATION_EVERY_N_EPOCHS": 1,
        "EARLY_STOPPING_PATIENCE": 6,
        "RANDAUGMENT_NUM_OPS": 2,
        "RANDAUGMENT_MAGNITUDE": 7,
        "NUM_WORKERS": 12,
        "PREFETCH": 8,
        "CACHE_TRAIN_SPLIT": True,
    }
    print(
        f"[ADAPTER_SELECTED] key={ADAPTER_KEY} crop={CROP_NAME} part={PART_NAME} "
        f"dataset={DATASET_NAME} ood={OOD_ROOT} oe_enabled={OE_ENABLED} oe={OE_ROOT} "
        f"run_id={RUN_ID}"
    )

    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())


[ADAPTER_SELECTED] key=strawberry__leaf crop=strawberry part=leaf dataset=strawberry__leaf ood=data/prepared_runtime_datasets/strawberry__leaf/ood oe_enabled=True oe=data/prepared_runtime_datasets/strawberry__leaf/oe run_id=strawberry_leaf_2026-05-06_06-00-15


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=strawberry epochs=12 bs=96 lr=0.0002 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=False
[DATASET] runtime_root=data/prepared_runtime_datasets dataset_name=strawberry__leaf
[OOD] ood_root=data/prepared_runtime_datasets/strawberry__leaf/ood ask_for_ood_root=False
[OE] oe_root=data/prepared_runtime_datasets/strawberry__leaf/oe ask_for_oe_root=False enabled=True loss_weight=0.15
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 
[OOD] factor=3.0 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [5]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [6]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


[OOD] explicit ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/ood
[OE] explicit oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/oe
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf classes=4: ['healthy', 'strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[DATASET][CHECK] scale=small continual=1606 val=491 test=491 ood=292 classes=4
[PARAMS] Manual overrides uygulandi: {'EPOCHS': 22, 'BATCH_SIZE': 96, 'LEARNING_RATE': 0.00015, 'LORA_R': 24, 'LORA_ALPHA': 24, 'LORA_DROPOUT': 0.1, 'OOD_FACTOR': 3.0, 'WEIGHT_DECAY': 0.01, 'LOSS_NAME': 'logitnorm', 'LOGITNORM_TAU': 1.0, 'MIXED_PRECISION': 'bf16', 'GRAD_ACCUM_STEPS': 1, 'VALIDATION_EVERY_N_EPOCHS': 1, 'EARLY_STOPPING_PATIENCE': 6, 'RANDAUGMENT_NUM_OPS': 2, 'RANDAUGMENT_MAGNITUDE': 7, 'NUM_WORKERS': 12, 'PREFETCH': 8, 'CACHE_TRAIN_SPLIT': True}
[PARAMS][FINAL] epochs=22 bs=96 lr=0.00015 lora

In [7]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


[OOD] selected ood root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/ood
[OE] selected oe root=/content/bitirmeprojesi/data/prepared_runtime_datasets/strawberry__leaf/oe
[CLASSES] ['healthy', 'strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=1 unmatched=3
[CLASSES] taxonomy-unmatched classes kept: ['strawberry_leaf_scorch_leaf', 'strawberry_leaf_spot_leaf', 'strawberry_powdery_mildew_leaf']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature": 1.0, "energy_temperature_mode": "auto", "energy_temperature_range": [0.5, 3.0], "energy_temperature_steps": 16, "knn_backend": "auto", "knn_chunk_size": 2048, "oe_enabled": true, "oe

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=13,770,248  classes=4


In [8]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.0}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [9]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


[TRAIN] epochs=22 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/16 loss=0.0000 lr=0.000150 speed=267.2s/s elapsed=8s eta=229s
[CKPT] epoch_end epoch=1 step=16
[EPOCH] 1/22: train_loss=1.4033 val_loss=1.3265 val_acc=0.1976 macro_f1=0.0880 * BEST
[TRAIN] epoch=2 batch=12/16 loss=0.0000 lr=0.000148 speed=266.6s/s elapsed=21s eta=244s
[CKPT] epoch_end epoch=2 step=32
[EPOCH] 2/22: train_loss=1.2999 val_loss=0.8922 val_acc=0.7006 macro_f1=0.5681 * BEST
[TRAIN] epoch=3 batch=12/16 loss=0.0000 lr=0.000144 speed=266.6s/s elapsed=33s eta=230s
[CKPT] epoch_end epoch=3 step=48
[EPOCH] 3/22: train_loss=0.9982 val_loss=0.4379 val_acc=0.9715 macro_f1=0.9686 * BEST
[TRAIN] epoch=4 batch=12/16 loss=0.0000 lr=0.000140 speed=265.9s/s elapsed=44s eta=215s
[CKPT] epoch_end epoch=4 step=64
[EPOCH] 4/22: train_loss=0.8516 val_loss=0.1552 val_acc=0.9552 macro_f1=0.9539 * BEST
[TRAIN] epoch=5 batch=12/16 loss=0.0000 lr=0.000134 speed=267.3s/s elapsed=56s eta=203s
[CKPT] epoch_end epoch=5 step=80
[EPO

In [10]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


[OOD] Kalibrasyon tamamlandi. classes=4 version=1


In [11]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/strawberry/leaf/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/strawberry_leaf_2026-05-06_06-00-15/artifacts/adapter_export/strawberry/leaf/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())


[DOGRULAMA (referans)] ornek=491 sinif=4 accuracy=0.9959 ood_auroc=0.9479 sure_ds_f1=0.7578 conformal_cov=0.9959
[TEST (ayrilmis)] ornek=491 sinif=4 accuracy=0.9939 ood_auroc=0.9399 sure_ds_f1=0.7610 conformal_cov=0.9939
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
